[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/708san/CV_data/blob/main/notebooks/colab/ocr_smoke_test.ipynb)

# CVライブラリ：日本語OCRのColab動作確認

このノートブックは、PaddleOCRの日本語OCRをCPUで試す確認用です。**別の新しいColabランタイムを開き、ランタイムのタイプをCPUにして、上から順に実行**してください。DETR・SAM用ノートブックと同じランタイムでは実行しません。

Notebook内で英数字の見本を作り、最後に公開済みの実レシート1枚でも推論します。実データの評価は部分注釈の範囲に限ります。PaddlePaddleとOCRモデルの取得に通信が発生します。Hugging Faceの公開モデルはログイン不要です。`HF_TOKEN`の警告は無視できます。以前のランタイムで古いcvlessonを読み込んだ場合は、ランタイムを削除してNotebookを再読み込みしてください。

In [ ]:
import sys
print("Python:", sys.version)
if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError("このライブラリの対応Pythonは3.11〜3.13です。Colabのランタイムを確認してください。")

## 1. OCR機能をインストール

CPUで使う場合は`ocr` extraだけでPaddlePaddle・PaddleOCR・OCRモデル用ライブラリが入ります。GPU版PaddlePaddleを使う場合だけは、OS/CUDAに合う公式GPU wheelを先に選んでください。


## 2. 公開版をインストール

PyPIのcv-data-lesson 0.1.0a6を導入します。新しいCPUランタイムで実行してください。OCRとPyTorch系機能を同じColab環境に追加する場合は依存の組み合わせが変わるため、機能ごとにランタイムを分ける方法が確実です。

In [ ]:
%pip install 'cv-data-lesson[ocr]==0.1.0a6'

In [ ]:
import inspect
from importlib.metadata import version
from cvlesson import ocr, display

source = inspect.getsource(ocr.load)
if "PP-OCRv5_mobile_det" not in source or "PP-OCRv5_mobile_rec" not in source:
    raise RuntimeError("古いcvlesson.ocrが読み込まれています。ランタイムを削除し、このNotebookを再読み込みして最初から実行してください。")
print("cvlesson OCR module:", ocr.__file__)
print("PaddlePaddle:", version("paddlepaddle"))
print("cvlesson devices:", ocr.available_devices())
assert ocr.available_devices()["cpu"] is True

## 3. テスト用の文字画像を作る

Pillowで「TOTAL 1280 JPY」を描きます。レシート画像や個人情報は使いません。

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

image = Image.new("RGB", (1200, 260), "white")
draw = ImageDraw.Draw(image)
font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = ImageFont.truetype(font_path, 92) if Path(font_path).exists() else ImageFont.load_default()
draw.text((40, 65), "TOTAL 1280 JPY", fill="black", font=font)
image_path = "/content/cvlesson_ocr_smoke.png"
image.save(image_path)
display(image)

## 4. OCRを実行

最初の読み込み時に固定版の日本語検出・認識モデルを取得します。結果を出し、文字列・信頼度・位置の形式を確認します。正確な文字列かどうかは、この動作確認では採点しません。

In [ ]:
reader = ocr.load(device="cpu")
result = reader.predict(image_path)
print("読み取り結果:")
print(result)  # OCRResultを文字列として読みやすく表示
print("行数:", len(result.lines))
assert isinstance(result.text, str)
for line in result.lines:
    assert isinstance(line.text, str)
    assert 0.0 <= line.confidence <= 1.0
    print(line.text, round(line.confidence, 3), line.polygon)
print("OCR推論と戻り値の確認: OK")
print("構造化データの例:", result.to_dict()["lines"][:1])
print("文字枠を重ねた画像:")
display.show(image, result, show_labels=False)  # 文字が多い画像は枠だけにすると見やすい


## 5. 公開実レシート画像で推論する

この段階ではCC BY-NC 4.0の公開済みの実レシート1枚を取得し、文字検出・読み取り処理が実画像で最後まで動くか確認します。非商用利用に限られます。画像や出力に店舗情報、QRコード、取引番号などが残る場合があるため、結果を公開・共有しないでください。

In [ ]:
from cvlesson import datasets

print(datasets.info("japanese_receipts_detailed"))  # CC BY-NC 4.0・非商用利用のみ
receipt_data = datasets.get("japanese_receipts_detailed")
receipt_frame = receipt_data.to_df()
sample = receipt_frame.iloc[0]
print("取得した実レシート画像:", len(receipt_frame), "枚 / split:", sample["split"])
print("部分注釈の形式:", type(sample["ground_truth"]).__name__)
real_result = reader.predict(sample["image_path"])
print("実画像OCRが完了:", bool(real_result.text), " / 読み取り行数:", len(real_result.lines))
print("注釈は画像内の一部領域のみです。全文OCRの正解ではないため、自動精度スコアは計算しません。")
display.show(sample["image_path"], real_result, show_labels=False)


## 結果の記録

- Python / PaddlePaddleの版：上の出力
- CPU推論が最後まで完了したか
- 読み取り結果と行数
- エラーが出た場合：最初に失敗したセルと全文を開発者へ共有

この結果はOCR APIが推論を実行できることの確認です。きれいな合成文字での結果を、レシートや手書き文字の精度保証に使わないでください。